# Project 3 — Predicting the All-In Cost of a Remittance

Regression model with a fairness audit. Mirrors the Credit Card Churn capstone structure:
Linear / RF / GradientBoosting / XGBoost → metrics → feature importance → fairness audit.

## Setup
```bash
pip install pandas numpy scikit-learn xgboost matplotlib seaborn openpyxl
```
Place `rpw_dataset_2011_2025_q1.xlsx` next to this notebook.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42
TARGET = 'Total cost (%) of transaction'

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'xgboost'

## Section 02 — Data

In [ ]:
df = pd.read_excel('rpw_dataset_2011_2025_q1.xlsx', sheet_name='Dataset (from Q2 2016)')
df = df.dropna(axis=1, how='all')
print('Raw shape:', df.shape)
df.head()

In [ ]:
# Clean target
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
df = df[df[TARGET].notna() & (df[TARGET] >= 0) & (df[TARGET] <= 50)].copy()
print(f'After cleaning target: {df.shape}, mean={df[TARGET].mean():.2f}%, median={df[TARGET].median():.2f}%')

# Drop low-volume corridors
ccount = df['corridor'].value_counts()
df = df[df['corridor'].isin(ccount[ccount >= 20].index)].copy()
print(f'After dropping low-volume corridors: {df.shape}, n_corridors={df["corridor"].nunique()}')

## Section 03 — Methodology / Feature Engineering

In [ ]:
df['log_send_usd'] = np.log1p(pd.to_numeric(df['Surveyed send amount (USD)'], errors='coerce').fillna(200))
firm_count = df.groupby(['corridor','period'])['firm'].nunique().rename('corridor_firm_count').reset_index()
df = df.merge(firm_count, on=['corridor','period'], how='left')

CAT_COLS = ['firm_type','payment instrument','Sending location','speed actual','pickup method',
            'source_region','destination_region','source_income','destination_income']
for c in CAT_COLS:
    df[c] = df[c].fillna('UNK').astype(str)

X_cat = pd.get_dummies(df[CAT_COLS], drop_first=True)
X_num = df[['log_send_usd','corridor_firm_count']].astype(float)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = df[TARGET].values
feature_names = list(X.columns)
print('Feature matrix:', X.shape)

In [ ]:
X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.20, random_state=RANDOM_STATE)
print(f'train={len(X_train):,}  test={len(X_test):,}')

## Section 04 — Models

In [ ]:
def evaluate(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    yp_tr = model.predict(Xtr); yp_te = model.predict(Xte)
    return model, yp_te, {
        'model': name,
        'rmse_train': np.sqrt(mean_squared_error(ytr, yp_tr)),
        'rmse_test':  np.sqrt(mean_squared_error(yte, yp_te)),
        'mae_test':   mean_absolute_error(yte, yp_te),
        'r2_train':   r2_score(ytr, yp_tr),
        'r2_test':    r2_score(yte, yp_te),
    }

# Model 01 - Linear baseline
lin, lin_pred, lin_m = evaluate('LinearRegression', LinearRegression(), X_train, X_test, y_train, y_test)
print(lin_m)

In [ ]:
# Model 02 - Random Forest with GridSearchCV
rf_grid = {'n_estimators':[200], 'max_depth':[None, 20], 'min_samples_leaf':[2]}
rf_search = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
                         rf_grid, scoring='r2', cv=5, n_jobs=-1)
rf_search.fit(X_train, y_train)
print('Best params:', rf_search.best_params_)
rf, rf_pred, rf_m = evaluate('RandomForest', rf_search.best_estimator_, X_train, X_test, y_train, y_test)
print(rf_m)

In [ ]:
# Model 03 - Gradient Boosting
gb, gb_pred, gb_m = evaluate('GradientBoosting',
    GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE),
    X_train, X_test, y_train, y_test)
print(gb_m)

In [ ]:
# Model 04 - XGBoost
xgb, xgb_pred, xgb_m = evaluate('XGBoost',
    XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.9,
                 colsample_bytree=0.9, reg_lambda=1.0, random_state=RANDOM_STATE,
                 n_jobs=-1, tree_method='hist'),
    X_train, X_test, y_train, y_test)
print(xgb_m)

## Section 05 — Results

In [ ]:
metrics = pd.DataFrame([lin_m, rf_m, gb_m, xgb_m])
metrics

## Section 06 — Feature importance

In [ ]:
def plot_imp(model, title):
    imp = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False).head(20)
    plt.figure(figsize=(8,6))
    sns.barplot(x=imp.values, y=imp.index, color='steelblue')
    plt.title(title); plt.xlabel('Feature importance'); plt.tight_layout(); plt.show()
plot_imp(rf, 'Random Forest — Top 20 Features')
plot_imp(xgb, 'XGBoost — Top 20 Features')

## Section 07 — Fairness audit (residuals by destination region)

In [ ]:
df_te = df_test.copy()
df_te['actual'] = y_test
df_te['pred']   = xgb_pred
df_te['residual'] = df_te['actual'] - df_te['pred']
summary = df_te.groupby('destination_region')['residual'].agg(['mean','median','std','count']).round(3).sort_values('mean', ascending=False)
summary

In [ ]:
order = summary.index.tolist()
plt.figure(figsize=(10,5))
sns.boxplot(data=df_te[df_te['destination_region'].isin(order)],
            x='destination_region', y='residual', order=order, showfliers=False, color='steelblue')
plt.axhline(0, ls='--', c='red', lw=1)
plt.xticks(rotation=30, ha='right')
plt.title('Fairness audit: cost prediction residuals by destination region\n(positive = pays more than predicted)')
plt.ylabel('Residual (actual - predicted) %')
plt.tight_layout(); plt.show()

## Section 08 — Action: Fair Price Calculator

Given a corridor + firm + instrument profile, what cost % does the model expect?
Anything above the predicted upper bound is flagged as overpriced.

In [ ]:
examples = df_test.head(15)[['corridor','firm','firm_type','payment instrument','speed actual',
                              'pickup method','Surveyed send amount (USD)']].copy()
examples['actual_cost_pct'] = y_test[:15].round(2)
examples['predicted_fair_cost_pct'] = xgb_pred[:15].round(2)
examples['flagged_overpriced'] = examples['actual_cost_pct'] > examples['predicted_fair_cost_pct'] * 1.25
examples